---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 4.1 — Prompt Optimization

---

## 💬 Product Check-in

> **Sam (Product):** "Eval scores have been stuck around the same range for a few iterations. We keep tweaking the system prompt by hand, but it feels like we're just rearranging words at this point."
>
> **Sam:** "Is there a more systematic way? Something automated?"

---

**Objective:** Use MLflow's prompt optimization API to automatically improve the system prompt using your existing eval dataset as the objective function.

In this notebook:
1. Load the latest evaluation dataset
2. Run the baseline eval with the current prompt
3. Run `optimize_prompts` to generate an improved prompt
4. Compare before and after scores

---
## 0 — Connect to MLflow

In [ ]:
from functools import cached_property
from pathlib import Path
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

ENV_FILE = Path.cwd().parent.parent / ".env"  # adjust .parent count to match your nesting

class AgentConfig(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=ENV_FILE,
        env_file_encoding="utf-8",
        extra="ignore",  # don't choke on unrelated vars in .env
    )

    # Secrets / endpoints (from .env)
    gemini_api_key: SecretStr
    gemini_openai_base_url: str

    # MLflow (from .env, with fallbacks)
    mlflow_tracking_uri: str
    experiment_name: str = Field(default="mlflow-agent-4", alias="EXPERIMENT_4_NAME")

    # Model knobs (hardcoded defaults, overridable via env if you want)
    llm: str = "gemini-2.5-flash-lite"
    temperature: float = 0.2
    max_tokens: int = 512
    ## Adding judge model to config
    judge_llm: str = "gemini:/gemini-3.1-flash-lite-preview"

    # Prompt
    prompt_name: str = "mlflow-agent-system"
    system_prompt: str = "You are an MLflow assistant."

    ##DATASET
    eval_dataset_name: str = "mlflow-agent-eval"


    ##ALIGN JUDGE MODEL TO CONFIG
    #Both the judge's assessments and human feedback must share the same assessment name 
    # on the same traces for alignment to work.
    align_judge: str = "response_quality"

    @cached_property
    def prompt_alias(self) -> str:
        return f"prompts:/{self.prompt_name}@prod"

cfg = AgentConfig()

In [ ]:
import os
import mlflow

if not os.getenv("MLFLOW_TRACKING_URI"):
    mlflow.set_tracking_uri("http://127.0.0.1:5000")

mlflow.set_experiment(cfg.experiment_name)
mlflow.openai.autolog()

---
## 1 — Loading the Eval Dataset

In experiment 1 we built up an evaluation dataset and persisted it as an MLflow `EvaluationDataset`. We continued to add examples to this dataset based on their effectiveness at replicating issues brought by Sam and showing resolution.

This dataset becomes our optimization objective — the prompt optimizer will try to maximize scores across *all* of these examples simultaneously.

In [ ]:
from mlflow.genai.datasets import get_dataset

eval_data = get_dataset(name=cfg.eval_dataset_name)

print(f"Loaded eval dataset: {len(eval_data.to_df())} records")
eval_data.to_df()[["inputs", "expectations"]].head()

---
## 2 — Baseline: Current Prompt Performance

Before optimizing, let's measure where the current prompt stands. This gives us a clear before/after comparison.

In [ ]:
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY_FREE"],
    base_url=os.environ["GEMINI_OPENAI_BASE_URL"],
)

prompt = mlflow.genai.load_prompt(cfg.prompt_alias)

print("Current prompt template:")
print("---")
print(prompt.template[:500])
print("---")

In [ ]:
from mlflow.types.llm import ChatMessage
from mlflow.types.llm import ChatCompletionResponse, ChatMessage, ChatParams

message = ChatMessage(role="user", content="What is MLflow?")

In [ ]:
from mlflow_langgraph_agent import agent
def predict_fn(question: str) -> str:
    message = ChatMessage(role="user", content=question)
    response = agent.predict(context="", messages=[message], params=ChatParams(temperature=cfg.temperature))
    return response


In [ ]:
predict_fn("What's the latest version of MLflow and its release date?")

In [ ]:
def mlflow_agent(question: str) -> str:
    response = openai_client.chat.completions.create(
        model=cfg.llm,
        messages=[
            {"role": "system", "content": "You are an MLflow assistant"},
            {"role": "user",   "content": question},
        ],
        temperature=cfg.temperature,
        max_completion_tokens=cfg.max_tokens,
    )
    return response.choices[0].message.content

In [ ]:
from scorers_4 import MLFLOW_SCORERS
from mlflow.genai.scorers import Correctness

baseline_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=MLFLOW_SCORERS
)

print("=== Baseline scores (current prompt) ===")
print(baseline_results.metrics)

> **Watch for it:** Note the baseline scores. The optimizer will try to improve across all scorers simultaneously. Even a small improvement across 7 diverse examples is meaningful — it means the prompt got better *in general*, not just on one cherry-picked example.

---
## 3 — How Prompt Optimization Works

MLflow's `optimize_prompts` API runs an automated optimization loop:

1. **Load** the current prompt from the registry
2. **Generate** candidate prompts using a reflection model that analyzes failures
3. **Evaluate** each candidate against the full dataset using your scorers
4. **Select** the best-performing candidate
5. **Register** the optimized prompt as a new version in the prompt registry

There are two optimizers available:

| Optimizer | How it works | Best for |
|---|---|---|
| `MetaPromptOptimizer` | Single-pass meta-prompting — fast, low data requirements | Quick iterations, smaller datasets |
| `GepaPromptOptimizer` | Genetic-Pareto iterative search — thorough, higher eval budget | Larger datasets, production-grade optimization |

We'll start with `MetaPromptOptimizer` for speed.

---
## 4 — Running Prompt Optimization

In [ ]:
from mlflow.genai.optimize.optimizers import MetaPromptOptimizer
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Guidelines

optimizer = MetaPromptOptimizer(
    reflection_model=f"gemini:/gemini-3-flash-preview", # Use different model from judges and agent
    guidelines="The prompt is for an MLflow assistant that helps developers with MLflow APIs, "
               "tracing, evaluation, and best practices. It should be concise, accurate, and "
               "politely redirect off-topic questions.",
)

# Refine guidelines for prompt optimizer
prompt = mlflow.genai.load_prompt(cfg.prompt_alias)

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=eval_data,
    prompt_uris=[prompt.uri],
    optimizer=optimizer,
    scorers=[Correctness(model=cfg.judge_llm)]
)

print("Optimization complete!")
print(f"Initial score: {result.initial_eval_score}")
print(f"Final score:   {result.final_eval_score}")

---
## 5 — Inspecting the Optimized Prompt

The optimizer registered a new version of the prompt in the MLflow prompt registry. Let's compare the original and optimized prompts side by side.

In [ ]:
optimized_prompt = result.optimized_prompts[0]

print("=" * 60)
print("ORIGINAL PROMPT")
print("=" * 60)
print(prompt.template)
print()
print("=" * 60)
print(f"OPTIMIZED PROMPT (version {optimized_prompt.version})")
print("=" * 60)
print(optimized_prompt.template)

Look at what changed. Common improvements the optimizer makes:
- Adding specificity about *how* to answer ("cite the function name and its key parameters")
- Adding explicit guardrails ("if the question is not about MLflow, politely redirect")
- Restructuring vague instructions into concrete behaviors
- Removing ambiguous phrasing that the model interprets inconsistently

---
## 6 — Verifying the Improvement

The optimizer reported scores, but let's run a clean evaluation with the optimized prompt to confirm.

In [ ]:
# Register prompt and set alieas as prod
# Replace 1 with the version of the optimized prompt
mlflow.genai.set_prompt_alias(name=cfg.prompt_name, version=1, alias="prod")

In [ ]:
optimized_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn, # Agent uses the prod prompt
    scorers=MLFLOW_SCORERS
)

print("=== Optimized prompt scores ===")
print(optimized_results.metrics)

---
## 3 — Adding the Tool to the Agent

With the Agents SDK, adding a tool is one line — pass it in the `tools` list. The SDK reads the `@function_tool` docstring and generates the JSON schema for the LLM automatically.

We also update the system prompt to tell the agent about the tool. This matters — the LLM needs to know *when* to use it.

In [ ]:
print("=== Before vs After ===")
print(f"{'Scorer':<30} {'Baseline':>10} {'Optimized':>10} {'Delta':>10}")
print("-" * 62)
for key in baseline_results.metrics:
    if key in optimized_results.metrics:
        before = baseline_results.metrics[key]
        after = optimized_results.metrics[key]
        delta = after - before
        sign = "+" if delta > 0 else ""
        print(f"{key:<30} {before:>10.3f} {after:>10.3f} {sign}{delta:>9.3f}")

The optimizer also logs per-scorer breakdowns:

In [ ]:
print("=== Per-scorer breakdown (from optimizer) ===")
print(f"{'Scorer':<30} {'Before':>10} {'After':>10}")
print("-" * 52)
for scorer_name in result.initial_eval_score_per_scorer:
    before = result.initial_eval_score_per_scorer[scorer_name]
    after = result.final_eval_score_per_scorer.get(scorer_name, 0)
    print(f"{scorer_name:<30} {before:>10.3f} {after:>10.3f}")

---
## 7 — The Iteration Loop

Prompt optimization isn't a one-shot fix. The pattern is:

1. **Eval** → find failures in the current prompt
2. **Optimize** → run `optimize_prompts` to generate a better candidate
3. **Verify** → re-run eval to confirm improvement
4. **Expand dataset** → add new examples that cover the failure modes you care about
5. **Repeat**

Let's run one more iteration with the `GepaPromptOptimizer` for a more thorough search.

In [ ]:
from mlflow.genai.optimize.optimizers import GepaPromptOptimizer

gepa_optimizer = GepaPromptOptimizer(
    reflection_model="gemini:/gemini-3-flash-preview", # Use different model from judges and agent,
    max_metric_calls=30,
)

# Use the optimized prompt as the starting point
result_v2 = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=eval_data,
    prompt_uris=[optimized_prompt.uri],
    optimizer=gepa_optimizer,
    scorers=MLFLOW_SCORERS
)

print("=== Second iteration (GEPA) ===")
print(f"Initial score: {result_v2.initial_eval_score}")
print(f"Final score:   {result_v2.final_eval_score}")

In [ ]:
final_prompt = result_v2.optimized_prompts[0]

print("=" * 60)
print(f"FINAL OPTIMIZED PROMPT (version {final_prompt.version})")
print("=" * 60)
print(final_prompt.template)

In [ ]:
def predict_fn_final(question: str) -> str:
    response = openai_client.chat.completions.create(
        model=cfg.llm,
        messages=[
            {"role": "system", "content": final_prompt.format()},
            {"role": "user", "content": question},
        ],
        temperature=cfg.temperature,
        max_completion_tokens=cfg.max_tokens,
    )
    return response.choices[0].message.content

final_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn_final,
    scorers=MLFLOW_SCORERS
)

print("=== Full comparison: Original → Meta → GEPA ===")
print(f"{'Scorer':<30} {'Original':>10} {'Meta':>10} {'GEPA':>10}")
print("-" * 62)
for key in baseline_results.metrics:
    if key in optimized_results.metrics and key in final_results.metrics:
        v0 = baseline_results.metrics[key]
        v1 = optimized_results.metrics[key]
        v2 = final_results.metrics[key]
        print(f"{key:<30} {v0:>10.3f} {v1:>10.3f} {v2:>10.3f}")

---
## 8 — Check the MLflow UI

Open the MLflow UI and look at the experiment. You should see:

1. **Evaluation runs** — one for each `mlflow.genai.evaluate()` call (baseline, optimized, final)
2. **Optimization runs** — logged by `optimize_prompts`, containing the candidate prompts tried and their scores
3. **Prompt registry** — new versions of your prompt, each tagged with the optimizer that created it

The prompt registry now contains the full lineage:

```
prompts:/mlflow-agent-system
│
├─ Version 1: Original hand-written prompt
├─ Version 2: Manual iteration (experiment 1)
├─ Version 3: MetaPromptOptimizer output
└─ Version 4: GepaPromptOptimizer output
```

Every version is traceable back to the eval run that validated it.

---
## Summary:

| Concept | How we used it |
|---|---|
| Product conversation | Sam's feedback → manual prompt tweaking has plateaued |
| `get_dataset()` | Loaded the 7-record eval dataset from experiment 1 as the optimization objective |
| Baseline eval | Measured current prompt performance before any optimization |
| `MetaPromptOptimizer` | Single-pass meta-prompting for a quick first improvement |
| `GepaPromptOptimizer` | Genetic-Pareto iterative search for a thorough second pass |
| `optimize_prompts()` | Automated the generate → evaluate → select loop |
| Prompt registry | Each optimized prompt registered as a new version with full lineage |
| Before/after comparison | Quantified improvement across all scorers |

### The pattern

```
Product feedback ("eval scores are stuck, manual tweaks aren't helping")
  → Load the eval dataset (the optimization objective)
  → Measure baseline with current prompt
  → Run optimize_prompts with MetaPromptOptimizer (fast first pass)
  → Verify improvement with a clean eval run
  → Run optimize_prompts with GepaPromptOptimizer (thorough second pass)
  → Compare Original → Meta → GEPA scores
  → Register best prompt, expand dataset, repeat
```

---

**Key insight:** The eval dataset *is* the optimization objective. A better evaluation dataset leads to a more optimial prompt. Every example you add to the dataset improves both your eval coverage *and* your prompt quality.